# 02 — Feature Extraction v2

Notebook này chạy pha feature extraction theo `feature_extraction_standard_v2.md`.

- Input: `data/processed_v4_rgb248_r4_exact/manifest.csv`
- Feature families: `always-on`, `conditional CFA`, `research-only`
- Core rule: notebook chỉ orchestration; toàn bộ logic trích xuất nằm trong `src/feature_extraction`.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_extraction import (
    ALL_FEATURE_KEYS,
    DEFAULT_CONFIG,
    load_feature_manifest,
    results_to_frame,
    run_feature_pipeline,
    save_feature_table,
    summarise_feature_table,
)

MANIFEST_PATH = PROJECT_ROOT / 'data' / 'processed_v4_rgb248_r4_exact' / 'manifest.csv'
MAX_FILES_ENV = os.getenv('FEATURE_EXTRACT_MAX_FILES', '').strip()
MAX_FILES = None if not MAX_FILES_ENV else int(MAX_FILES_ENV)
WORKERS = int(os.getenv('FEATURE_EXTRACT_WORKERS', str(min(8, os.cpu_count() or 4))))
FORCE_RERUN = os.getenv('FEATURE_EXTRACT_FORCE_RERUN', '0') == '1'
SHOW_PROGRESS = os.getenv('FEATURE_EXTRACT_SHOW_PROGRESS', '0') == '1'
RUN_NAME = 'feature_extraction_v2_rgb248_exact' if MAX_FILES is None else f'feature_extraction_v2_rgb248_exact_smoke_{MAX_FILES}'
OUTPUT_CSV = PROJECT_ROOT / 'features' / (f'{RUN_NAME}.csv')
AUDIT_ROOT = PROJECT_ROOT / 'audit_output' / 'validation' / RUN_NAME
SUMMARY_PATH = AUDIT_ROOT / 'feature_extraction_summary.json'
AUDIT_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG = DEFAULT_CONFIG

print({'manifest': str(MANIFEST_PATH), 'max_files': MAX_FILES, 'workers': WORKERS, 'output_csv': str(OUTPUT_CSV), 'feature_version': CONFIG.feature_version})

{'manifest': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\data\\processed_v4_rgb248_r4_exact\\manifest.csv', 'max_files': 16, 'workers': 1, 'output_csv': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\features\\feature_extraction_v2_rgb248_exact_smoke_16.csv', 'feature_version': 'v2_rgb248_exact_multibranch'}


## 1. Load accepted preprocessing manifest

Cell này chỉ đọc manifest preprocessing v4, lọc `ACCEPTED`, gán `split_role`, và tùy chọn lấy sample smoke theo `FEATURE_EXTRACT_MAX_FILES`.

In [2]:
manifest = load_feature_manifest(MANIFEST_PATH, config=CONFIG, max_files=MAX_FILES)
manifest[['generator', 'label', 'split_role', 'patch_path']].head(10)

,generator,label,split_role,patch_path
0,ADM,ai,id_test,C:\Users\USER\Desktop\ai_detector_img\data\pro...
1,ADM,ai,train_core,C:\Users\USER\Desktop\ai_detector_img\data\pro...
2,ADM,nature,train_core,C:\Users\USER\Desktop\ai_detector_img\data\pro...
3,ADM,nature,train_core,C:\Users\USER\Desktop\ai_detector_img\data\pro...
4,GLIDE,ai,ood_eval,C:\Users\USER\Desktop\ai_detector_img\data\pro...
5,GLIDE,nature,ood_eval,C:\Users\USER\Desktop\ai_detector_img\data\pro...
6,Midjourney,ai,train_core,C:\Users\USER\Desktop\ai_detector_img\data\pro...
7,Midjourney,nature,val,C:\Users\USER\Desktop\ai_detector_img\data\pro...
8,SDv14,ai,train_core,C:\Users\USER\Desktop\ai_detector_img\data\pro...
9,SDv14,nature,val,C:\Users\USER\Desktop\ai_detector_img\data\pro...


## 2. Run or load feature extraction

Nếu file output đã tồn tại và `FORCE_RERUN=False`, notebook sẽ load lại. Nếu không, notebook sẽ chạy full extraction bằng API package.

In [3]:
if OUTPUT_CSV.exists() and not FORCE_RERUN:
    feature_frame = pd.read_csv(OUTPUT_CSV)
else:
    results = run_feature_pipeline(
        manifest,
        config=CONFIG,
        workers=WORKERS,
        chunksize=32,
        show_progress=SHOW_PROGRESS,
    )
    feature_frame = results_to_frame(results, config=CONFIG)
    save_feature_table(feature_frame, OUTPUT_CSV)
summary = summarise_feature_table(feature_frame, config=CONFIG)
summary.update({'run_name': RUN_NAME, 'output_csv': str(OUTPUT_CSV), 'max_files': MAX_FILES, 'workers': WORKERS})
SUMMARY_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
summary

{'rows': 16,
 'ok_rows': 16,
 'error_rows': 0,
 'feature_count': 36,
 'split_role_counts': {'train_core': 8, 'ood_eval': 4, 'val': 3, 'id_test': 1},
 'generator_counts': {'ADM': 4,
  'GLIDE': 2,
  'Midjourney': 2,
  'SDv14': 2,
  'SDv15': 2,
  'VQDM': 2,
  'Wukong': 2},
 'cfa_validity_score': {'mean': -0.7977088205098058,
  'std': 0.34960118022945813,
  'q10': -0.9842333750119286,
  'q50': -0.6680479037734487,
  'q90': -0.543728099268346},
 'run_name': 'feature_extraction_v2_rgb248_exact_smoke_16',
 'output_csv': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\features\\feature_extraction_v2_rgb248_exact_smoke_16.csv',
 'max_files': 16,
 'workers': 1}

## 3. Status and split QA

Kiểm tra nhanh trạng thái extraction, số hàng theo split, và shape output.

In [4]:
feature_frame.groupby(['split_role', 'status']).size().unstack(fill_value=0)

status,ok
split_role,
id_test,1
ood_eval,4
train_core,8
val,3


## 4. Feature preview

Xem một số cột quan trọng của nhánh `always-on` và `conditional`.

In [5]:
preview_cols = [
    'generator', 'label', 'split_role', 'status',
    'frs_mid_variance', 'fft_mid_logenergy', 'spatial_snr_ratio',
    'cfa_rg_pi_xy', 'cfa_bg_pi_xy', 'cfa_validity_score'
]
feature_frame[preview_cols].head(12)

,generator,label,split_role,status,frs_mid_variance,fft_mid_logenergy,spatial_snr_ratio,cfa_rg_pi_xy,cfa_bg_pi_xy,cfa_validity_score
0,ADM,ai,id_test,ok,0.968468,-0.737449,0.859031,0.011070,0.011613,-0.497814
1,ADM,ai,train_core,ok,0.703509,-0.467041,0.980570,0.000448,0.000203,-0.660379
2,ADM,nature,train_core,ok,0.333926,-0.101671,0.437233,0.001055,0.001672,-0.531332
3,ADM,nature,train_core,ok,0.403029,-0.175713,0.627193,0.003692,0.006626,-0.657740
4,GLIDE,ai,ood_eval,ok,1.379989,-0.594683,0.855497,0.003166,0.004492,-2.007402
5,GLIDE,nature,ood_eval,ok,0.079943,0.027363,1.096130,0.004739,0.001880,-0.556124
6,Midjourney,ai,train_core,ok,0.161177,-0.390692,1.244917,0.017324,0.000671,-0.657295
7,Midjourney,nature,val,ok,0.714174,-0.324534,0.491995,0.001012,0.002360,-0.739735
8,SDv14,ai,train_core,ok,0.863080,-0.410124,1.400359,0.004410,0.007566,-0.615625
9,SDv14,nature,val,ok,0.534053,-0.406303,0.751190,0.003471,0.007480,-0.961525


## 5. Conditional CFA validity summary

Cell này chỉ xem phân bố `cfa_validity_score` để phục vụ bước audit/gating phía sau.

In [6]:
feature_frame['cfa_validity_score'].describe()

count    16.000000
mean     -0.797709
std       0.361067
min      -2.007402
25%      -0.938494
50%      -0.668048
75%      -0.610461
max      -0.497814
Name: cfa_validity_score, dtype: float64